# 第六课（最后一课）：拆解 STGCN 与图结构消融

前面已经分别理解时间模型和图消息传递。本课把它们组合起来，并回答一个真正的实验问题：

> STGCN 的预测来自网络容量，还是来自真实道路图提供的空间关系？

本课完成：逐层张量追踪、时间门控卷积、完整 STGCN 前后向、单 batch 过拟合，以及真实图/单位阵/打乱图的公平消融。

## 0. 实验定义先于代码

三组模型具有完全相同的结构和初始参数，只改变邻接矩阵：

- **Real graph**：真实道路图；
- **Identity graph**：节点只看自己，相当于关闭空间消息；
- **Shuffled graph**：保持图的边权、稀疏度和度结构，但破坏传感器与图节点的对应关系。

Real 优于 Identity 表明节点通信有帮助；Real 进一步优于 Shuffled，才说明真实空间对应关系有额外价值。单个种子只能用于学习和初步诊断，正式结论仍需 3 个种子。

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle  # noqa: E402
from crosscity.data.graph import load_adjacency, normalize_adjacency  # noqa: E402
from crosscity.evaluation.metrics import horizon_metrics, masked_mae  # noqa: E402
from crosscity.models.stgcn import GraphConv, STGCN, TemporalGatedConv  # noqa: E402
from crosscity.utils import seed_everything  # noqa: E402

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = load_config(ROOT / 'configs/metr_la.yaml')
bundle = build_data_bundle(config.dataset)
raw_a = load_adjacency(config.dataset.adjacency_path)
real_a = normalize_adjacency(raw_a)
identity_a = torch.eye(raw_a.shape[0])
print('device:', device, '| nodes:', raw_a.shape[0])

### 关于第五课发现的重复自环

METR-LA 原始图已有对角线 1。项目的 `normalize_adjacency` 现已先清除原对角线，再统一添加一次自环。因此旧 checkpoint 是在“双自环”定义下训练的，不能与本课结果直接混为同一实验。旧结果仍可作为开发记录，新消融应全部采用修正后的统一定义。

In [ ]:
raw_without_diagonal = raw_a.copy()
np.fill_diagonal(raw_without_diagonal, 0)
expected = normalize_adjacency(raw_without_diagonal)
print('有/无原始自环，归一化后最大差:', (real_a - expected).abs().max().item())
print('归一化后对角线范围:', real_a.diag().min().item(), real_a.diag().max().item())
assert torch.equal(real_a, expected)

## 1. 时间门控卷积在做什么？

输入在 STGCN 内部采用 `[B,C,T,N]`。卷积核形状为 `(K,1)`：沿时间轴看相邻时刻，但不沿节点轴滑动。因此它不会自行混合不同传感器。

模块把卷积输出拆成 value 和 gate：

$$Y=(\mathrm{value}+\mathrm{residual})\odot\sigma(\mathrm{gate})$$

sigmoid 门控制每个位置保留多少信息；残差路径使梯度和原始信号更容易传递。

In [ ]:
xb, yb, mb = next(iter(DataLoader(bundle.train, batch_size=2, shuffle=False)))
internal_x = xb.permute(0, 3, 1, 2)
temporal = TemporalGatedConv(in_channels=1, out_channels=8)
temporal_out = temporal(internal_x)
print('dataset x :', xb.shape, '= [B,T,N,F]')
print('internal x:', internal_x.shape, '= [B,F,T,N]')
print('temporal y:', temporal_out.shape, '= [B,C,T,N]')
print('卷积权重:', temporal.conv.weight.shape, '= [2*C,C_in,K,1]')

### 局部性实验

只扰动一个节点，时间卷积的其他节点输出不应改变；图卷积之后，具有连接关系的节点才可能受到影响。

In [ ]:
base = torch.zeros(1, 1, 12, 207)
changed = base.clone()
changed[:, :, 5, 16] = 1.0
temporal.eval()
with torch.no_grad():
    temporal_difference = (temporal(changed) - temporal(base)).abs()
affected_temporal_nodes = torch.flatnonzero(temporal_difference.sum(dim=(0, 1, 2)) > 0)
print('时间卷积后受影响的节点:', affected_temporal_nodes.tolist())
assert affected_temporal_nodes.tolist() == [16]

In [ ]:
graph = GraphConv(channels=8)
graph.eval()
with torch.no_grad():
    before_graph = temporal(changed)
    after_graph_real = graph(before_graph, real_a)
    after_graph_identity = graph(before_graph, identity_a)
graph_difference = (after_graph_real - after_graph_identity).abs().sum(dim=(0, 1, 2))
print('真实图与只看自己产生差异的节点数:', int((graph_difference > 0).sum()))
print('差异最大的 10 个节点:', torch.topk(graph_difference, 10).indices.tolist())

## 2. 完整 STGCN 的信息流

项目结构为：

```text
[B,1,12,N]
 → TemporalGatedConv → GraphConv → TemporalGatedConv → BatchNorm
 → TemporalGatedConv → GraphConv → TemporalGatedConv → BatchNorm
 → 跨越全部12个历史步的卷积 → 12步预测头
 → [B,12,N]
```

两次 GraphConv 意味着一个节点最多可以间接接收两跳邻居的信息。

In [ ]:
seed_everything(SEED)
model = STGCN(input_steps=12, output_steps=12, hidden_dim=32)
shapes = {}
hooks = []
for name, module in model.named_modules():
    if isinstance(module, (TemporalGatedConv, GraphConv, torch.nn.BatchNorm2d)):
        def remember(module, inputs, output, name=name):
            shapes[name] = tuple(output.shape)
        hooks.append(module.register_forward_hook(remember))

with torch.no_grad():
    prediction = model(xb, real_a)
for hook in hooks:
    hook.remove()
display(pd.Series(shapes, name='output shape').to_frame())
print('最终预测:', prediction.shape, '| target:', yb.shape)
assert prediction.shape == yb.shape

hook 只观察中间张量，不修改计算。阅读形状时始终追踪四个轴：batch、channel、time、node。节点数始终保留到最后，时间维在预测头中由 12 压缩成 1，输出 channel 则代表未来 12 步。

In [ ]:
parameter_groups = {
    'temporal convolutions': 0,
    'graph channel transforms': 0,
    'normalization and head': 0,
}
for name, parameter in model.named_parameters():
    count = parameter.numel()
    if 'temporal_' in name:
        parameter_groups['temporal convolutions'] += count
    elif '.graph.' in name:
        parameter_groups['graph channel transforms'] += count
    else:
        parameter_groups['normalization and head'] += count
pd.Series(parameter_groups).to_frame('parameters').assign(share=lambda frame: frame.parameters / frame.parameters.sum())

邻接矩阵不是 `nn.Parameter`，因此不进入 checkpoint。跨城市时迁移的是时间卷积、通道变换、归一化和预测头权重；目标城市在前向计算中提供自己的邻接矩阵。

## 3. 单 batch 过拟合：检查完整链路

固定 batch 包含多个窗口和全部 207 个节点，所以不要求损失逼近零；重点是它明显下降，证明时间层、图层、预测头、mask 和梯度能够协同工作。

In [ ]:
seed_everything(SEED)
tiny_model = STGCN(12, 12, hidden_dim=32).to(device)
tiny_x, tiny_y, tiny_mask = [item.to(device) for item in next(iter(DataLoader(bundle.train, batch_size=8, shuffle=False)))]
tiny_a = real_a.to(device)
optimizer = torch.optim.Adam(tiny_model.parameters(), lr=1e-2)
tiny_losses = []
for step in range(201):
    optimizer.zero_grad()
    loss = masked_mae(tiny_model(tiny_x, tiny_a), tiny_y, tiny_mask)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(tiny_model.parameters(), 5.0)
    optimizer.step()
    tiny_losses.append(loss.item())
print('initial loss:', tiny_losses[0])
print('final loss  :', tiny_losses[-1])
plt.plot(tiny_losses)
plt.xlabel('step')
plt.ylabel('normalized masked MAE')
plt.title('STGCN fixed-batch check')
plt.show()

## 4. 构造严格的三组图

Shuffled graph 通过同时重排真实邻接矩阵的行列来保持图的内部结构，却不重排速度列。因此图节点与真实传感器错位。若特征也一起重排，就只是改编号，不再是破坏空间关系的消融。

In [ ]:
generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(real_a.shape[0], generator=generator)
shuffled_a = real_a[permutation][:, permutation]
graphs = {'real': real_a, 'identity': identity_a, 'shuffled': shuffled_a}
graph_checks = pd.DataFrame({
    name: {
        'nonzero': int((adjacency > 0).sum()),
        'sum': adjacency.sum().item(),
        'max': adjacency.max().item(),
    } for name, adjacency in graphs.items()
}).T
graph_checks

Real 与 Shuffled 的统计量应相同，区别仅是哪些传感器相连。Identity 的非零项只有 N 个。

## 5. 公平训练消融

先用 FAST 跑通；再设为 FULL。三组训练前都会恢复完全相同的初始权重并重置随机种子，使 DataLoader shuffle 顺序一致。早停只看 validation，最后统一评估 test。

In [ ]:
FAST_MODE = True  # 跑通后改为 False，从这里重新运行到结果分析
BATCH_SIZE = 64
MAX_EPOCHS = 6 if FAST_MODE else 50
PATIENCE = 3 if FAST_MODE else 8

def limited(dataset, maximum):
    return Subset(dataset, range(min(len(dataset), maximum)))

train_set = limited(bundle.train, 2048) if FAST_MODE else bundle.train
val_set = limited(bundle.val, 1024) if FAST_MODE else bundle.val
test_set = limited(bundle.test, 2048) if FAST_MODE else bundle.test
print('实际 train/val/test:', len(train_set), len(val_set), len(test_set))

In [ ]:
@torch.no_grad()
def evaluate(model, dataset, adjacency):
    model.eval()
    predictions, targets, masks = [], [], []
    for xb, yb, mb in DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False):
        predictions.append(model(xb.to(device), adjacency).cpu())
        targets.append(yb)
        masks.append(mb)
    pred = bundle.scaler.inverse_transform(torch.cat(predictions))
    target = bundle.scaler.inverse_transform(torch.cat(targets))
    return horizon_metrics(pred, target, torch.cat(masks))

def validation_mae(model, dataset, adjacency):
    return evaluate(model, dataset, adjacency)['overall']['mae']

In [ ]:
def train_one_graph(name, adjacency, initial_state):
    seed_everything(SEED)
    model = STGCN(12, 12, hidden_dim=32).to(device)
    model.load_state_dict(initial_state)
    adjacency = adjacency.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_val, best_state, stale = float('inf'), None, 0
    records = []
    loader_generator = torch.Generator().manual_seed(SEED)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total, count = 0.0, 0
        loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator)
        for xb, yb, mb in loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            optimizer.zero_grad()
            loss = masked_mae(model(xb, adjacency), yb, mb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += loss.item() * int(mb.sum())
            count += int(mb.sum())
        train_loss = total / count
        val_mae = validation_mae(model, val_set, adjacency)
        records.append({'graph': name, 'epoch': epoch, 'train_scaled_mae': train_loss, 'val_mae': val_mae})
        print(f'{name:8s} epoch {epoch:02d} | train-z {train_loss:.4f} | val-mph {val_mae:.4f}')
        if val_mae < best_val:
            best_val, stale = val_mae, 0
            best_state = deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= PATIENCE:
                print(name, 'early stopping')
                break
    model.load_state_dict(best_state)
    return model, records, evaluate(model, test_set, adjacency)


训练日志中的 train 是标准化空间 MAE，validation 是反标准化后的 mph MAE，因此数值不能直接相减；它们分别用于观察优化和选择 checkpoint。

In [ ]:
seed_everything(SEED)
initial_model = STGCN(12, 12, hidden_dim=32)
initial_state = deepcopy(initial_model.state_dict())
trained_models, histories, metrics = {}, [], {}
for name, adjacency in graphs.items():
    trained_models[name], records, metrics[name] = train_one_graph(name, adjacency, initial_state)
    histories.extend(records)
print('三组训练完成。')

In [ ]:
history_frame = pd.DataFrame(histories)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, group in history_frame.groupby('graph'):
    axes[0].plot(group.epoch, group.train_scaled_mae, marker='o', label=name)
    axes[1].plot(group.epoch, group.val_mae, marker='o', label=name)
axes[0].set(title='Training loss', xlabel='epoch', ylabel='normalized masked MAE')
axes[1].set(title='Validation performance', xlabel='epoch', ylabel='MAE (mph)')
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. 阅读结果，而不只找最小数字

In [ ]:
mae_table = pd.DataFrame({name: {horizon: values['mae'] for horizon, values in result.items()} for name, result in metrics.items()})
mae_table = mae_table.loc[['horizon_3', 'horizon_6', 'horizon_12', 'overall']]
mae_table.round(3).style.highlight_min(axis=1, color='lightgreen')

In [ ]:
gain = pd.DataFrame(index=mae_table.index)
gain['real vs identity (%)'] = (mae_table.identity - mae_table.real) / mae_table.identity * 100
gain['real vs shuffled (%)'] = (mae_table.shuffled - mae_table.real) / mae_table.shuffled * 100
gain.round(2)

正增益表示 Real 更好，负增益表示 Real 更差。请分 horizon 阅读：图可能只对中长期传播有用，也可能在所有时距都造成过平滑。FAST 结果不用于研究结论；FULL 单种子仍只是初步证据。

In [ ]:
metric_rows = []
for graph_name, result in metrics.items():
    for horizon, values in result.items():
        metric_rows.append({'graph': graph_name, 'horizon': horizon, **values})
result_frame = pd.DataFrame(metric_rows)
output_path = ROOT / 'artifacts/notebook06_graph_ablation.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
result_frame.to_csv(output_path, index=False)
print('结果已保存:', output_path)

## 7. 如何把单种子升级为正式实验

正式结果不能只反复运行 notebook。应将图消融加入配置驱动命令，然后对种子 42、43、44 分别训练，保存每次 checkpoint 与指标，最后报告均值±标准差。图策略是一个实验变量，不能在看到 test 后再决定保留哪个策略。

跨城市阶段则固定使用同样的自环规则，依次比较 scratch、zero-shot、full-finetune、temporal-frozen 和 target-full。目标城市始终使用自己的邻接矩阵。

## 8. 最终验收问题

1. 时间卷积为什么不会让节点 16 影响其他节点？哪个模块负责传播？
2. 两个 STBlock 为什么可以带来两跳感受野？
3. 为什么旧的双自环 checkpoint 不应直接放进本课消融表？
4. Identity 图与 Shared MLP/LSTM 是否完全相同？为什么？
5. Shuffled 图为何必须只重排邻接矩阵，而不能同步重排速度特征？
6. Real 优于 Identity、但不优于 Shuffled，可以得出什么有限结论？
7. 为什么三组模型要使用相同初始化和 DataLoader 顺序？
8. 单种子 Real 最优，能否写成“真实空间图显著提升性能”？还缺什么？

请带回：单 batch 初末 loss、FULL 三组 MAE 表、两张增益列、训练曲线现象，以及最不确定的一题。

至此，六课完成了从数据窗口、mask、基线、时间模型、图消息传递到 STGCN 消融的完整基础。下一阶段不再增加教学模型，而是运行双城市、三预算、三种子的迁移实验，并把结果整理成研究报告。